In [ ]:
!git clone https://github.com/fashn-AI/fashn-vton-1.5.git
%cd fashn-vton-1.5
!pip install -q -e .
!pip install -q fastapi uvicorn python-multipart

In [ ]:
!python scripts/download_weights.py --weights-dir ./weights

In [ ]:
!pip install -q rembg

In [ ]:
%%writefile tryon_server.py
import io
import os
import queue
import threading
import time
import uuid

from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.responses import Response
from PIL import Image
from fashn_vton import TryOnPipeline
from rembg import remove, new_session

SERVICE_KEY = os.environ["TRYON_SERVICE_KEY"]
VALID_CATEGORIES = {"tops", "bottoms", "one-pieces"}
MAX_BYTES = 5 * 1024 * 1024
JOB_TTL_SECONDS = 30 * 60
BACKGROUND_COLOR = (235, 235, 235)  # light gray; change here if you want another shade

app = FastAPI()
pipeline = TryOnPipeline(weights_dir="./weights")
cutout_session = new_session("u2net")  # loaded once at startup

jobs = {}
jobs_lock = threading.Lock()
work_queue = queue.Queue()


def check_key(key):
    if key != SERVICE_KEY:
        raise HTTPException(401, "Unauthorized")


def read_image(upload: UploadFile) -> Image.Image:
    data = upload.file.read(MAX_BYTES + 1)
    if len(data) > MAX_BYTES:
        raise HTTPException(413, "Image too large")
    try:
        return Image.open(io.BytesIO(data)).convert("RGB")
    except Exception:
        raise HTTPException(400, "Invalid image")


def light_gray_background(img: Image.Image) -> Image.Image:
    # Cut out the person and place them on a plain light gray background
    try:
        cutout = remove(img, session=cutout_session).convert("RGBA")
        background = Image.new("RGBA", cutout.size, BACKGROUND_COLOR + (255,))
        background.alpha_composite(cutout)
        return background.convert("RGB")
    except Exception as error:
        print("BACKGROUND ERROR (using original):", str(error))
        return img


def cleanup_old_jobs():
    now = time.time()
    with jobs_lock:
        expired = [j for j, v in jobs.items() if now - v["created"] > JOB_TTL_SECONDS]
        for job_id in expired:
            del jobs[job_id]


def worker():
    # One job at a time, so the GPU is never overloaded
    while True:
        job_id, person_img, garment_img, category = work_queue.get()
        with jobs_lock:
            jobs[job_id]["status"] = "running"
        try:
            result = pipeline(
                person_image=person_img,
                garment_image=garment_img,
                category=category,
            )
            final_img = light_gray_background(result.images[0])
            buffer = io.BytesIO()
            final_img.save(buffer, format="PNG")
            with jobs_lock:
                jobs[job_id].update(status="done", result=buffer.getvalue())
        except Exception as error:
            # Log the error only, never the images
            print("INFERENCE ERROR:", job_id, str(error))
            with jobs_lock:
                jobs[job_id].update(status="error")
        finally:
            work_queue.task_done()


threading.Thread(target=worker, daemon=True).start()


@app.get("/health")
def health():
    return {"ok": True}


@app.post("/try-on")
def start_try_on(
    person: UploadFile = File(...),
    garment: UploadFile = File(...),
    category: str = Form("tops"),
    x_service_key: str = Header(None),
):
    check_key(x_service_key)
    cleanup_old_jobs()

    if category not in VALID_CATEGORIES:
        raise HTTPException(400, "Invalid category")

    person_img = read_image(person)
    garment_img = read_image(garment)

    job_id = uuid.uuid4().hex
    with jobs_lock:
        jobs[job_id] = {"status": "queued", "created": time.time(), "result": None}
    work_queue.put((job_id, person_img, garment_img, category))

    return {"job_id": job_id}


@app.get("/try-on/{job_id}")
def job_status(job_id: str, x_service_key: str = Header(None)):
    check_key(x_service_key)
    with jobs_lock:
        job = jobs.get(job_id)
    if not job:
        raise HTTPException(404, "Job not found")
    return {"status": job["status"]}


@app.get("/try-on/{job_id}/result")
def job_result(job_id: str, x_service_key: str = Header(None)):
    check_key(x_service_key)
    with jobs_lock:
        job = jobs.get(job_id)
        if not job:
            raise HTTPException(404, "Job not found")
        if job["status"] != "done":
            raise HTTPException(409, "Not ready")
        image_bytes = job["result"]
        del jobs[job_id]  # free memory once the result is picked up
    return Response(content=image_bytes, media_type="image/png")

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

In [ ]:
import os, re, subprocess, time

SERVICE_KEY = "your-TRYON_SERVICE_KEY-here"
env = dict(os.environ, TRYON_SERVICE_KEY=SERVICE_KEY)

# Stop anything left from a previous run
!pkill -f cloudflared
!pkill -f uvicorn
time.sleep(2)

server_log = open("/content/server.log", "w")
server = subprocess.Popen(
    ["uvicorn", "tryon_server:app", "--port", "8000"],
    env=env, cwd="/content/fashn-vton-1.5",
    stdout=server_log, stderr=subprocess.STDOUT,
)

# Wait until the server answers (loading the model can take a few minutes)
ready = False
for _ in range(120):
    if server.poll() is not None:
        break
    check = subprocess.run(["curl", "-s", "http://localhost:8000/health"], capture_output=True, text=True)
    if check.stdout.strip():
        ready = True
        break
    time.sleep(3)

if not ready:
    print("SERVER FAILED TO START. Last lines of the log:")
    print(open("/content/server.log").read()[-3000:])
else:
    print("Server is running on port 8000")

    tunnel_log = open("/content/tunnel.log", "w")
    tunnel = subprocess.Popen(
        ["/content/cloudflared", "tunnel", "--url", "http://localhost:8000"],
        stdout=tunnel_log, stderr=subprocess.STDOUT,
    )

    public_url = None
    for _ in range(60):
        time.sleep(1)
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
        if match:
            public_url = match.group(0)
            break

    print("Public URL:", public_url)

In [ ]:
!curl -s http://localhost:8000/health; echo
!ps aux | grep -E "uvicorn|cloudflared" | grep -v grep
!ls /content/fashn-vton-1.5/weights

In [ ]:
   !curl -s -m 10 http://localhost:8000/health; echo

In [ ]:
   print(len(SERVICE_KEY))

In [ ]:
   !nvidia-smi

In [ ]:
   import subprocess
   print(subprocess.run(["curl", "-s", "-H", f"x-service-key: {SERVICE_KEY}",
       "http://localhost:8000/try-on/80bdaa5177f94a4eb27025a17221ba87"],
       capture_output=True, text=True).stdout)

In [ ]:
   !tail -n 40 /content/server.log